# SICSS @ Penn 2026 Tutorial: Human Mobility Science (Part 2)

TA: Jorge Francisco Barreras (fbarrer@sas.upenn.edu)
## Notebook setup

In [ ]:
!pip install esda
!pip install splot
!pip install -U gdown
!pip install folium
!pip install git+https://github.com/Watts-Lab/nomad.git@darren-contact-sip

In [ ]:
# Load the prepared Philadelphia tutorial data.
data_url = "https://drive.google.com/drive/folders/1Y88OdsiyidywcXTcSXlsXi86d85Uv9q8?usp=drive_link"
input_path = Path("data")

gdown.download_folder(
    url=data_url,
    output=str(input_path),
    use_cookies=False,
)

#### Import packages

In [ ]:
from esda.moran import Moran
import folium
import geopandas as gpd
from libpysal.weights import Queen
import matplotlib
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'monospace'
import numpy as np
from pathlib import Path
import pandas as pd
import nomad.io.base as loader
from nomad.stop_detection.viz import plot_pings, plot_circles
from nomad.contact_estimation import estimate_contacts, compute_contact_weights
from nomad.metrics.metrics import (
    rog as nomad_rog,
    self_containment as nomad_self_containment,
    social_interaction_potential,
)
from scipy.stats import pearsonr
from splot.esda import moran_scatterplot
import time

#### Choropleth helper
This helper creates an interactive choropleth map used later. You can run it and ignore the output.

In [ ]:
def choropleth_folium(gdf, key_col, val_col, m = None, fill_cmap = 'YlGnBu'):
    gdf = gdf.to_crs(4326)
    if m == None:
      m = folium.Map(width = 850, height = 500, location=[gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()], zoom_start=10,tiles="Cartodb Positron")
    folium.Choropleth(
        geo_data=gdf.to_json(),
        data=gdf,
        columns=[key_col, val_col],
        key_on=f'feature.properties.{key_col}',
        fill_color=fill_cmap,
        fill_opacity=0.7,
        line_opacity=0.2,
        legend_name=val_col
    ).add_to(m)
    return m


## Initial data exploration

Our primary *input* is a *stop/activity* table from a synthetic Philadelphia mobility dataset generated using NOMAD.[1]

- Sparse trajectories are sampled device pings
- Complete trajectories are full simulated paths
- Diaries describe generated activity schedules
- Stops are the located activity intervals used below for radius of gyration, self-containment, and SIP.

First, look at the columns that we have.

- **userid**: unique identifier representing one synthetic user in our dataset
- **start_time**: the time at which the activity started
- **end_time**: the time at which the activity ended
- **duration**: duration of the activity, in minutes
- **loc_area**: Philadelphia Census Block Group where the activity took place
- **home_area**: Philadelphia Census Block Group associated with the user's home location
- **activity_type**: activity label inferred from the synthetic location type, such as Home, Work, Retail, or Park
- **o_lon, o_lat**: projected coordinates of the activity location in EPSG:3857; these are meter-based projected coordinates used for distance calculations
- **o_h7, o_h8, o_h9**: H3 cells at resolutions 7, 8, and 9 containing the activity location
- **home_activity**: `'Y'` if the activity is at the user's home, `'N'` otherwise
- **geometry**: point geometry associated with the activity location

You can see the first 5 rows of the data below.

[1] Li, Thomas H., and Francisco Barreras. "Garden city: A synthetic dataset and sandbox environment for analysis of pre-processing algorithms for GPS human mobility data." arXiv preprint arXiv:2412.00913 (2024).

In [ ]:
# Load Philadelphia Census Block Groups.
philly_areas = gpd.read_file(input_path / "philly_areas.geojson")

stops = loader.from_file(
    input_path / "df_samp.parquet",
    format="parquet",
    traj_cols=stop_cols,
    user_id="userid",
    x="o_lon",
    y="o_lat",
    start_datetime="start_time",
    end_datetime="end_time")

# Turn into geopandas by assigning a geometry column
travel_data = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(stops["o_lon"], stops["o_lat"]),
    crs="EPSG:3857",
)

travel_data["x"] = travel_data["o_lon"]
travel_data["y"] = travel_data["o_lat"]
# small fix
travel_data.loc[(travel_data.loc_area != travel_data.home_area), "home_activity"] = "N"

print(f"Loaded {len(travel_data):,} precomputed stops for {travel_data.userid.nunique():,} synthetic users.")
print("-------------------------------------------------------")
travel_data.head()

**Note**: The coordinate columns `o_lon` and `o_lat` are projected x-y coordinates in EPSG:3857 (Web Mercator), not decimal-degree longitude/latitude. 

Now, let's look at how much data we have for each user.

In [ ]:
fig,ax=plt.subplots(figsize=(7,5))

# Histogram of activities observed per user.
activities_peruser = travel_data.groupby('userid').size()
total_days_peruser = travel_data.groupby('userid').agg({'start_time':lambda x: x.dt.date.nunique()}).start_time
activities_perday = activities_peruser/total_days_peruser

ax.hist(activities_perday,bins=30,edgecolor='black')
ax.set_ylabel('Number of users')
ax.set_xlabel('Activities per day')
plt.show()

This distribution gives a quick sense of how many stop records we observe per user per day.

What does this data actually look like? **Let's look at some sample users --- how do they move around Philadelphia?**

I have picked two sample users for us to compare, but if you want to randomly select two new users, you can do so by uncommenting the second group of lines in the code chunk below.

We'll plot their observed activities on a map of Philadelphia. **Each data point is an activity or stop at a single location.** The two users are shown with different marker colors and shapes, and home activities are highlighted in **green**.

In [ ]:
sample_user = 'clever_noether'
sample_user2 = 'focused_thompson'

# # UNCOMMENT THE LINES BELOW TO SELECT NEW, RANDOM USERS
# sample_user, sample_user2 = travel_data.userid.drop_duplicates().sample(2).tolist()

sample_user_data = travel_data[travel_data.userid==sample_user].copy()
sample_user_data2 = travel_data[travel_data.userid==sample_user2].copy()


#### Map sample users with NOMAD

The map below uses NOMAD's plotting helpers to show the two sample users' stops over Philadelphia Census Block Groups.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

base_areas = philly_areas.to_crs(travel_data.crs)

plot_pings(
    sample_user_data,
    ax=ax,
    color="cornflowerblue",
    s=20,
    alpha=0.75,
    base_geometry=base_areas,
    base_geom_color="lightgrey",
    base_geom_background="white",
    traj_cols=stop_cols,
)
plot_pings(
    sample_user_data2,
    ax=ax,
    color="orange",
    marker="s",
    s=20,
    alpha=0.75,
    traj_cols=stop_cols,
)

for user_data, color, marker in [
    (sample_user_data, "darkgreen", "o"),
    (sample_user_data2, "darkgreen", "s"),
]:
    plot_pings(
        user_data[user_data["home_activity"] == "Y"],
        ax=ax,
        color=color,
        marker=marker,
        s=45,
        edgecolor="black",
        linewidth=0.6,
        traj_cols=stop_cols,
    )

legend_handles = [
    matplotlib.lines.Line2D([], [], color="cornflowerblue", marker="o", linestyle="None", label="Sample user 1 stops"),
    matplotlib.lines.Line2D([], [], color="orange", marker="s", linestyle="None", label="Sample user 2 stops"),
    matplotlib.lines.Line2D([], [], color="darkgreen", marker="o", markeredgecolor="black", linestyle="None", label="Home stops"),
]
ax.legend(handles=legend_handles, loc="upper right")
ax.set_title("Precomputed Philadelphia stops loaded with NOMAD")
ax.set_axis_off()


The two users show different activity spaces: one has stops concentrated more centrally/westward, while the other has stops extending farther east/southeast. Their activity spaces are not completely separate; some observed stops are close to one another.

**What are some ways that we can quantify our sample users' mobility patterns?** In the next section, we calculate mobility statistics that can be used to compare them.


## Calculate Metrics

We will calculate three mobility characteristics:

1. **Radius of gyration**: the size of a user's activity space.
2. **Self-containment**: how often a user stays close to home.
3. **Social interaction potential**: potential exposure to other users in nearby places at overlapping times.

We start with radius of gyration, then use the same pattern for the other metrics.


### Radius of gyration

Radius of gyration measures the spread of a user's stops around their average location.

$$
R_g = \sqrt{\frac{1}{N_{\text{stay}}} \sum_{i=1}^{N_{\text{stay}}} \left[(P_i^x - \bar{P}^x)^2 + (P_i^y - \bar{P}^y)^2\right]}
$$

Here, $(\bar{P}^x, \bar{P}^y)$ is the user's average stop location. Larger values mean a wider breadth of movement.


We start by calculating radius of gyration for two sample users. Then we compute for every user, inspect the distribution, and aggregate users by home area to look for spatial patterns.


In [ ]:
metric_cols = {
    "user_id": "userid",
    "datetime": "start_time",
    "x": "x",
    "y": "y",
    "duration": "duration",
    "location_id": "home_activity",
}

def radius_gyration(data):
    """Compute radius of gyration using NOMAD's ROG metric helper."""
    rog_df = nomad_rog(
        data,
        agg_freq="D",
        weighted=True,
        traj_cols=metric_cols,
        exploded=False,
    )
    return rog_df["rog"].mean()

start = time.time()
print(f"Radius of gyration, user 1: {np.round(radius_gyration(sample_user_data), 3)} meters")
print(f"Radius of gyration, user 2: {np.round(radius_gyration(sample_user_data2), 3)} meters")
print(f"Calculation time: {time.time() - start:.3f} seconds")

The two sample users have different activity-space sizes. The map below draws each user's stops and adds a circle with radius equal to that user's radius of gyration.


In [ ]:
sample_user_data = travel_data[travel_data.userid == sample_user].copy()
sample_user_data2 = travel_data[travel_data.userid == sample_user2].copy()

fig, ax = plt.subplots(figsize=(7, 7))

rog_centers = gpd.GeoDataFrame(
    {"rad_gyr": [radius_gyration(sample_user_data), radius_gyration(sample_user_data2)]},
    geometry=[sample_user_data.geometry.union_all().centroid, sample_user_data2.geometry.union_all().centroid],
    crs=travel_data.crs,
)

plot_circles(
    rog_centers.iloc[[0]],
    ax=ax,
    radius="rad_gyr",
    color="darkblue",
    edgecolor="darkblue",
    alpha=0.15,
    base_geometry=base_areas,
    base_geom_color="lightgrey",
    base_geom_background="white",
)
plot_circles(
    rog_centers.iloc[[1]],
    ax=ax,
    radius="rad_gyr",
    color="chocolate",
    edgecolor="chocolate",
    alpha=0.15,
)
plot_pings(sample_user_data, ax=ax, color="cornflowerblue", s=18, alpha=0.75, traj_cols=stop_cols)
plot_pings(sample_user_data2, ax=ax, color="orange", marker="s", s=18, alpha=0.75, traj_cols=stop_cols)
plot_pings(rog_centers.iloc[[0]], ax=ax, color="darkblue", s=45)
plot_pings(rog_centers.iloc[[1]], ax=ax, color="chocolate", marker="s", s=45)

legend_handles = [
    matplotlib.lines.Line2D([], [], color="cornflowerblue", marker="o", linestyle="None", label="Sample user 1 stops"),
    matplotlib.lines.Line2D([], [], color="orange", marker="s", linestyle="None", label="Sample user 2 stops"),
    Patch(facecolor="darkblue", edgecolor="darkblue", alpha=0.15, label="Sample user 1 ROG"),
    Patch(facecolor="chocolate", edgecolor="chocolate", alpha=0.15, label="Sample user 2 ROG"),
]
ax.legend(handles=legend_handles, loc="upper right")
ax.set_title("Radius of gyration for sample users")
ax.set_axis_off()


We might be curious: what are some underlying drivers of differences in radius of gyration? For example, do users whose homes are closer to study-area center tend to have smaller or larger activity spaces? To begin answering that kind of question, we first calculate radius of gyration for all users in our stop table.

#### Compute radius of gyration for all users

Calculate the same metric for every user in the stop table. The result is `rad_gyr_df`, a user-level table that we can summarize and map.


In [ ]:
# Apply the radius of gyration function to the full dataset.
rad_gyr_df = nomad_rog(
    travel_data,
    agg_freq="D",
    weighted=True,
    traj_cols=metric_cols,
    exploded=False,
).rename(columns={"rog": "rad_gyr"})

rad_gyr_df = (
    rad_gyr_df
    .groupby("userid", as_index=False)
    .agg(rad_gyr=("rad_gyr", "mean"))
)

# Add users' home information.
homes_df = (
    travel_data[travel_data.home_activity == "Y"]
    .groupby("userid")
    .agg(
        home_lat=("o_lat", "mean"),
        home_lon=("o_lon", "mean"),
        home_area=("home_area", "first"),
    )
    .reset_index()
)

rad_gyr_df = rad_gyr_df.merge(homes_df,on='userid')
rad_gyr_df["rad_gyr"] = rad_gyr_df.rad_gyr.round()
rad_gyr_df.head(12)


#### Explore the user-level distribution

Let's explore what the distribution of radius of gyration looks like across our sample users.

First, take a look at the table we've just created:
- **userid**: unique user id
- **home_lat**: y-coordinate of user's inferred home location
- **home_lon**: x-coordinate of user's inferred home location
- **home_area**: Philadelphia Census Block Group associated with the user's home location
- **rad_gyr**: radius of gyration, in meters


The histogram below shows the distribution of radius of gyration across users. The two sample users are marked in blue and orange; the mean and median are marked in black.


In [ ]:
plt.hist(rad_gyr_df.rad_gyr,bins=30,color='lightgrey',edgecolor='grey',alpha=.5)
plt.axvline(rad_gyr_df[rad_gyr_df.userid==sample_user].rad_gyr.values[0],
            color='darkblue',
            linewidth=1.5,
            label='Sample user 1')
plt.axvline(rad_gyr_df[rad_gyr_df.userid==sample_user2].rad_gyr.values[0],
            color='chocolate',
            linewidth=1.5,
            label='Sample user 2')
plt.axvline(rad_gyr_df.rad_gyr.mean(),
            color='black',
            linewidth=2,
            linestyle='-.',
            label=f'Mean: {np.round(rad_gyr_df.rad_gyr.mean()/1000,2)}km')
plt.axvline(rad_gyr_df.rad_gyr.median(),
            color='black',
            linewidth=2,
            linestyle='dotted',
            label=f'Median: {np.round(rad_gyr_df.rad_gyr.median()/1000,2)}km')
plt.legend()
plt.title('Radius of gyration')

The median radius of gyration is indicated with a dotted line above. We will use this median as a threshold in the self-containment section below.

#### Explore spatial patterns by home area

Next, aggregate user-level radius of gyration to each user's home Census Block Group. This lets us ask whether activity-space size varies across Philadelphia neighborhoods.

The area-level table contains:
- **area_id**: Philadelphia Census Block Group id
- **rad_gyr**: average radius of gyration for users whose home is in that area
- **dist_from_center**: distance from the area's centroid to the study-area center


In [ ]:
area_rad_gyr = (
    rad_gyr_df
    .groupby("home_area", as_index=False)
    .agg(rad_gyr=("rad_gyr", "mean"), users=("userid", "nunique"))
)

# To join spatially we need to be in the same projection
philly_areas_projected = philly_areas.to_crs(travel_data.crs)
rad_gyr_areas = philly_areas_projected.merge(
    area_rad_gyr,
    left_on="area_id",
    right_on="home_area",
    how="inner",
)

philly_center = rad_gyr_areas.geometry.union_all().centroid
rad_gyr_areas["dist_from_center"] = rad_gyr_areas.geometry.centroid.distance(philly_center)
rad_gyr_areas.head()


**Does radius of gyration vary across space?**

The map below shows average radius of gyration by home Census Block Group.


In [ ]:
m = choropleth_folium(rad_gyr_areas, "area_id", "rad_gyr")
m


One simple check is whether users farther from the study-area center tend to have larger activity spaces.


In [ ]:
plt.scatter(rad_gyr_areas.dist_from_center, rad_gyr_areas.rad_gyr, s=8)
plt.xlabel("Distance from study-area center (meters)")
plt.ylabel("Average radius of gyration (meters)")

r = pearsonr(rad_gyr_areas.dist_from_center, rad_gyr_areas.rad_gyr)
print(f"Pearson correlation: {np.round(r.statistic, 3)}")
print(f"p-value: {np.round(r.pvalue, 3)}")


Finally, Moran's I tests for spatial autocorrelation---whether nearby areas tend to have similar values.


In [ ]:
w = Queen.from_dataframe(rad_gyr_areas, use_index=False)
w.transform = "r"

moran = Moran(rad_gyr_areas["rad_gyr"], w)
fig, ax = moran_scatterplot(moran)
ax.set_xlabel("Radius of gyration")
ax.set_ylabel("Spatial lag")

print(f"Moran's I: {np.round(moran.I, 3)}")
print(f"p-value: {np.round(moran.p_sim, 3)}")


Moran's I summarizes spatial clustering. A positive value means nearby areas tend to have similar average radius of gyration; a value near zero means little spatial pattern.


## Self-containment and social interaction potential

### Self-containment

Self-containment describes **the propensity of individuals to stay close to home**. It is calculated as the percentage of non-home activities that are closer to home than the median radius of gyration in a given sample:

$$S_c^{non-home} = \frac{\sum_{i\in \text{non-residence activities}} \theta_i}{N_{\text{non-residence activities}}}$$

$$\theta_i = \begin{cases}1 \text{ if } \sqrt{(P_i^x - P_{home}^x)^2 + (P_i^y - P_{home}^y)^2} \leq Median(R_g)\\0 \text{ otherwise} \end{cases}$$

We use NOMAD’s self-containment helper to calculate the share of non-home stops within the median radius of gyration from home.

#### Self-containment workflow

We use the median radius of gyration as a distance threshold, compute each user's share of nearby non-home stops, visualize that threshold for the two sample users, and then inspect the user-level distribution.


In [ ]:
metric_cols = {
    "user_id": "userid",
    "datetime": "start_time",
    "x": "x",
    "y": "y",
    "duration": "duration",
    "location_id": "home_activity",
}

rg_med = np.quantile(rad_gyr_df.rad_gyr, 0.5)

self_df_all = nomad_self_containment(
    travel_data,
    threshold=rg_med,
    agg_freq="D",
    weighted=True,
    home_activity_type="Y", # The string that identifies the user's home in the column location_id
    traj_cols=metric_cols,
    exploded=False,
).rename(columns={"self_containment": "cont_nonhome"})

self_df = (
    self_df_all
    .groupby("userid", as_index=False)
    .agg(cont_nonhome=("cont_nonhome", "mean"))
)

sample_self = self_df[self_df.userid.isin([sample_user, sample_user2])]
sample_self

#### Visualize the distance threshold

The two sample users differ in their self-containment values, reflecting how often their non-home stops fall within the median radius of gyration from home.

In the map below, each grey circle is centered on a user's home location and has radius equal to the sample median radius of gyration. Stops inside the circle count toward the self-containment measure, while stops outside the circle represent activities farther from home. Comparing the two users shows how the same distance threshold can summarize different activity-space patterns.

In [ ]:
home_pt = sample_user_data[sample_user_data.home_activity=='Y'].geometry.values[0]
sample_user_data.loc[:,'dist_from_home'] = sample_user_data.geometry.distance(home_pt)
home_pt2 = sample_user_data2[sample_user_data2.home_activity=='Y'].geometry.values[0]
sample_user_data2.loc[:,'dist_from_home'] = sample_user_data2.geometry.distance(home_pt2)

home_points = gpd.GeoDataFrame(geometry=[home_pt, home_pt2], crs=travel_data.crs)

fig, ax = plt.subplots(figsize=(7, 7))

plot_circles(
    home_points,
    ax=ax,
    radius=rg_med,
    color="silver",
    edgecolor="dimgray",
    alpha=0.4,
    linewidth=2,
    base_geometry=base_areas,
    base_geom_color="lightgrey",
    base_geom_background="white",
)
plot_pings(sample_user_data, ax=ax, color="cornflowerblue", s=16, alpha=0.45, traj_cols=stop_cols)
plot_pings(sample_user_data[sample_user_data.dist_from_home<rg_med], ax=ax, color="darkblue", s=22, alpha=0.85, traj_cols=stop_cols)
plot_pings(sample_user_data2, ax=ax, color="orange", marker="s", s=16, alpha=0.45, traj_cols=stop_cols)
plot_pings(sample_user_data2[sample_user_data2.dist_from_home<rg_med], ax=ax, color="chocolate", marker="s", s=22, alpha=0.85, traj_cols=stop_cols)
plot_pings(home_points, ax=ax, color="darkgreen", s=45, edgecolor="black", linewidth=0.6)

legend_handles = [
    matplotlib.lines.Line2D([], [], color="cornflowerblue", marker="o", linestyle="None", label="Sample user 1 stops"),
    matplotlib.lines.Line2D([], [], color="orange", marker="s", linestyle="None", label="Sample user 2 stops"),
    matplotlib.lines.Line2D([], [], color="darkblue", marker="o", linestyle="None", label="Within threshold"),
    matplotlib.lines.Line2D([], [], color="darkgreen", marker="o", markeredgecolor="black", linestyle="None", label="Home"),
]
ax.legend(handles=legend_handles, loc="upper right")
ax.set_title("Self-containment threshold for sample users")
ax.set_axis_off()


In [ ]:
self_df.head()

#### Explore the self-containment distribution

Let's explore the distribution of self-containment across users.

The table contains:
- **userid**: unique user id
- **cont_nonhome**: share of non-home stops within the median radius of gyration


The histogram below shows how much users vary in the share of non-home stops that remain close to home.


In [ ]:
plt.hist(self_df.cont_nonhome,bins=60,color='lightgrey',edgecolor='grey',alpha=.5)
plt.axvline(self_df[self_df.userid==sample_user].cont_nonhome.values[0],
            color='darkblue',
            linewidth=1.5,
            label='Sample user 1')
plt.axvline(self_df[self_df.userid==sample_user2].cont_nonhome.values[0],
            color='chocolate',
            linewidth=1.5,
            label='Sample user 2')
plt.axvline(self_df.cont_nonhome.mean(),
            color='dimgrey',
            linewidth=2,
            linestyle='-.',
            label=f'Mean: {np.round(self_df.cont_nonhome.mean(),3)*100}%')
plt.axvline(self_df.cont_nonhome.median(),
            color='dimgrey',
            linewidth=2,
            linestyle='dotted',
            label=f'Median: {np.round(self_df.cont_nonhome.median(),3)*100}%')
plt.legend()
plt.title('Self-containment (%)')


## Social interaction potential

Social interaction potential estimates how much opportunity users have to encounter one another. It is a metric based on a co-location network, requiring a relatively expensive join to *estimate stops that overlap in space and time*. These overlaps can be weighted by duration and distance, and aggregated for analysis.

#### Estimate nearby overlapping stops

`estimate_contacts` returns pairs of users whose stops overlap in time and are within a chosen distance threshold. The output includes the two users, the contact interval, overlap duration, and distance.

In [ ]:
max_distance = 100

sip_cols = {
    **stop_cols,
    "start_datetime": "start_time",
    "end_datetime": "end_time",
}

sip_users = travel_data.userid.drop_duplicates().sample(200, random_state=1)

sip_stops = (
    travel_data[
        (travel_data.userid.isin(sip_users)) &
        (travel_data.duration >= 5)
    ]
    .copy()
)

print(f"Running SIP on {sip_stops.userid.nunique():,} users and {len(sip_stops):,} stops.")
start = time.time()
contacts = estimate_contacts(
    sip_stops,
    distance_threshold=max_distance,
    traj_cols=sip_cols,
)
print(f"Found {len(contacts):,} contact events in {time.time() - start:.2f} seconds")
contacts.head(20)

#### Weight contact events

Here we use a linear distance weight: longer overlaps count more, and closer contacts receive more weight. Contacts at the distance threshold receive zero weight.

In [ ]:
contacts["contact_weight"] = compute_contact_weights(
    contacts,
    method="linear_distance",
    distance_threshold=max_distance,
)

contacts[[
    "user_id_1",
    "user_id_2",
    "overlap_duration",
    "distance",
    "contact_weight",
]].head(10)

#### Aggregate weighted contacts into SIP

`social_interaction_potential` credits each contact to both users and sums the contact weights. We also calculate a same-home-area version as a simple tutorial grouping variable.

In [ ]:
sip_total = social_interaction_potential(
    contacts,
    weight_col="contact_weight",
).rename(columns={"user_id": "userid", "sip": "sip_total"})

user_home_area = travel_data.groupby("userid")["home_area"].first()
contacts["home_area_1"] = contacts.user_id_1.map(user_home_area)
contacts["home_area_2"] = contacts.user_id_2.map(user_home_area)

sip_same_home_area = social_interaction_potential(
    contacts[contacts.home_area_1 == contacts.home_area_2],
    weight_col="contact_weight",
).rename(columns={"user_id": "userid", "sip": "sip_same_home_area"})

sip_df = sip_total.merge(sip_same_home_area, on="userid", how="left")
sip_df["sip_same_home_area"] = sip_df["sip_same_home_area"].fillna(0)
sip_df["sip_pct_same_home_area"] = sip_df.sip_same_home_area / sip_df.sip_total
sip_df.head()

#### Explore same-home-area exposure

The histogram below shows the share of each user's weighted exposure that came from contacts with users assigned to the same home Census Block Group.


In [ ]:
plt.hist(sip_df.sip_pct_same_home_area.dropna(), bins=30, color="lightgrey", edgecolor="grey", alpha=0.7)
plt.xlabel("Share of weighted exposure with users from the same home area")
plt.ylabel("Number of users")
plt.title("Social interaction potential")
plt.show()

The same-home-area share is a simple teaching example. In an applied analysis, this grouping variable could be replaced with more meaningful census-based strata.

## Further exercises

1. Recalculate radius of gyration with duration weights. How much do long stops change the result?
2. Change the SIP distance threshold. How does the number of contacts change?
3. Compare self-containment using all stops versus non-home stops only. What changes conceptually?
4. Discuss which preprocessing choices could affect these mobility metrics before the data reaches this notebook.